In [ ]:
# =====================================================================
# build_package_colab.py
# Run in ONE Colab cell (Python runtime). Before running:
#   1. Upload ssl-cxr-bibliometric.zip into /MyDrive/bibliometrik-cxr/
#   2. Paste this whole file into a new cell and run it.
# Output: /MyDrive/bibliometrik-cxr/ssl-cxr-bibliometric-final.zip
# =====================================================================
import os, glob, shutil, zipfile, json
from google.colab import drive
drive.mount('/content/drive')

PROJ = "/content/drive/MyDrive/bibliometrik-cxr"
SKEL = os.path.join(PROJ, "ssl-cxr-bibliometric.zip")
WORK = "/content/pkg"
REPO = os.path.join(WORK, "ssl-cxr-bibliometric")
os.chdir(PROJ)
shutil.rmtree(WORK, ignore_errors=True); os.makedirs(WORK)
zipfile.ZipFile(SKEL).extractall(WORK)
missing, added = [], []

def find(patterns):
    out = []
    for p in patterns:
        out += [f for f in glob.glob(p, recursive=True) if not f.startswith("Rlib")]
    return sorted(set(out))

def copy(patterns, dest, label):
    fs = find(patterns)
    if not fs: missing.append(label); return
    os.makedirs(os.path.join(REPO, dest), exist_ok=True)
    for f in fs:
        shutil.copy(f, os.path.join(REPO, dest, os.path.basename(f))); added.append(f)

import pandas as pd

# 1. Screening decisions (no titles/abstracts)
try:
    s = pd.read_excel("03_clean/screening.xlsx")
    cols = {"EID": "EID", "DOI": "DOI", "Year": "year", "keputusan": "decision",
            "alasan": "code", "tugas": "task"}
    s = s[[c for c in cols if c in s.columns]].rename(columns=cols)
    s.to_csv(os.path.join(REPO, "02_screening/screening_decisions.csv"), index=False)
    added.append("screening_decisions.csv  " + str(s["decision"].value_counts().to_dict()))
except Exception as e:
    missing.append(f"screening_decisions.csv ({e})")

# 2. Extraction matrix (coding only)
try:
    m = pd.read_excel("03_clean/matriks_ekstraksi.xlsx")
    m = m.drop(columns=[c for c in ["AU", "TI", "AB", "SO", "DE", "ID"] if c in m.columns])
    m.to_csv(os.path.join(REPO, "06_extraction/extraction_matrix.csv"), index=False)
    added.append(f"extraction_matrix.csv ({len(m)} rows)")
except Exception as e:
    missing.append(f"extraction_matrix.csv ({e})")

# 3. Thesauri
copy(["03_clean/thesaurus_vos*.txt", "**/thesaurus_vos*.txt"], "03_thesaurus", "thesaurus_vos.txt")
copy(["03_clean/thesaurus_cocitation*.txt", "**/thesaurus_cocitation*.txt"], "03_thesaurus", "thesaurus_cocitation.txt")

# 4. Notebooks with outputs cleared
nbs = find(["04_notebooks/*.ipynb", "**/0[1-8]_*.ipynb"])
if not nbs: missing.append("notebooks 01-08")
os.makedirs(os.path.join(REPO, "04_notebooks"), exist_ok=True)
for f in nbs:
    nb = json.load(open(f, encoding="utf-8"))
    for c in nb.get("cells", []):
        if c.get("cell_type") == "code":
            c["outputs"] = []; c["execution_count"] = None
    nb.get("metadata", {}).pop("widgets", None)
    json.dump(nb, open(os.path.join(REPO, "04_notebooks", os.path.basename(f)), "w",
                       encoding="utf-8"), indent=1)
    added.append(f + " (outputs cleared)")

# 5. Figures, VOSviewer files, burst sensitivity
copy(["05_figures/*.png"], "05_results/figures", "figures (05_figures/*.png)")
copy(["**/*map*.txt", "**/*network*.txt", "05_figures/vos_*.txt"], "05_results/vosviewer",
     "VOSviewer map/network files")
copy(["05_figures/*burst*sens*.csv", "05_figures/*burst*.csv", "05_figures/*burst*.xlsx"],
     "05_results", "burst sensitivity")

# 6. Safety: remove anything licensed by Scopus that slipped in
for f in glob.glob(REPO + "/**/*", recursive=True):
    b = os.path.basename(f).lower()
    if b.startswith("scopus_s1") or b.startswith("corpus_") or b in ("m.rds", "raw.rds", "screening.xlsx"):
        os.remove(f); print("removed (licensed data):", f)
os.remove(os.path.join(REPO, "build_package_colab.py")) if os.path.exists(os.path.join(REPO, "build_package_colab.py")) else None

# 7. Zip
out = os.path.join(PROJ, "ssl-cxr-bibliometric-final.zip")
shutil.make_archive(out[:-4], "zip", WORK, "ssl-cxr-bibliometric")
print("\nADDED:"); [print("  ", a) for a in added]
print("\nMISSING (optional, add by hand if you have them):"); [print("  ", x) for x in missing]
print("\nDone:", out)